# **Análisis de Datos Saber 11 - Departamento de Caldas**

Este notebook implementa el proceso de selección, limpieza, alistamiento y análisis exploratorio de los datos de las pruebas Saber 11 para el departamento de Caldas, como parte del Proyecto 2 del curso *Analítica Computacional para la Toma de Decisiones*. El producto final está orientado al **Ministerio de Educación** como usuario final, y busca responder tres preguntas de negocio relacionadas con equidad socioeconómica, desempeño territorial y brechas de género.

## Tarea 2 - Selección, limpieza y alistamiento de datos

Los datos provienen del portal de [Datos Abiertos de Colombia](https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe), actualizados a abril de 2024. La selección, limpieza y alistamiento de datos ya fue realizada en el Proyecto 1.

## Tarea 3 - Exploración y análisis de datos

Con los datos limpios y correctamente tipados, se realiza un análisis de datos orientado a responder las tres preguntas de negocio que hemos desarrollado. Esto incluye estadísticas descriptivas, histogramas, diagramas de caja, diagramas de dispersión y mapas de calor sobre distintas variables relevantes para el desarrollo de los modelos de redes neuronales en tarea4/.

---

Daniel Benavides - 202220428 

Juanita Cortés - 202222129 

Andrés Felipe Herrera - 202220888

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from prettytable import PrettyTable

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

alt.data_transformers.enable('vegafusion')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Carga de datos

In [ ]:
df = pd.read_csv('../data/caldas_data.csv')

print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
df.head(5)

## 1. Pregunta de negocio

### *¿Cuál es el puntaje global esperado para un estudiante de Caldas dado su perfil socioeconómico y el tipo de institución educativa?*

## 2. Pregunta de negocio

### *¿Puede identificarse si un estudiante está en riesgo de obtener un puntaje global por debajo del umbral de bajo desempeño, según sus características socioeconómicas y escolares?*

Esta pregunta se responde con un **modelo de clasificación binaria**. La variable objetivo es:

$$\text{bajo\_rendimiento} = \begin{cases} 1 & \text{si } \texttt{punt\_global} < 250 \\ 0 & \text{en otro caso} \end{cases}$$

El umbral de 250 puntos es consistente con el análisis del Proyecto 1, donde se identificó ese valor como frontera entre municipios de rendimiento crítico y rendimiento medio-alto.

### 2.1 Feature Engineering

In [ ]:
UMBRAL_BAJO = 230

# --- Target variable ---
# Se define la variable objetivo como clasificación binaria
# Se eliminan filas sin punt_global ya que es indispensable para el target

df = df.dropna(subset=['punt_global'])
df['bajo_rendimiento'] = (df['punt_global'] < UMBRAL_BAJO).astype(int)

print(f"Distribución del target:\n{df['bajo_rendimiento'].value_counts()}")
print(f"\n% bajo rendimiento: {df['bajo_rendimiento'].mean():.1%}")

In [ ]:
# --- Estrato: ordinal numérico 1-6 ---
# El P1 mostró que el estrato tiene efecto progresivo y monotónico sobre el puntaje,
# por lo que es apropiado tratarlo como ordinal numérico en lugar de dummies.
estrato_map = {
    'Estrato 1': 1, 'Estrato 2': 2, 'Estrato 3': 3,
    'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6
}
df['estrato_num'] = df['fami_estratovivienda'].map(estrato_map)

# --- Educación de los padres: ordinal numérico ---
# El P1 mostró un gradiente continuo: a mayor educación, mayor puntaje.
# Asignamos un entero que preserva el orden natural del nivel educativo.
# 'No sabe' / 'No aplica' → NaN para no introducir ruido.
edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9,
}

df['edu_madre_num'] = df['fami_educacionmadre'].map(edu_map)   # NaN si 'No sabe'/'No aplica'
df['edu_padre_num'] = df['fami_educacionpadre'].map(edu_map)

# --- Índice de activos del hogar (0-4) ---
# Creado en P1. Si no existe, se recalcula.
asset_cols = ['fami_tienecomputador', 'fami_tieneinternet',
              'fami_tieneautomovil', 'fami_tienelavadora']
if 'indice_activos' not in df.columns:
    df['indice_activos'] = df[asset_cols].sum(axis=1, skipna=True)

# --- Variables escolares binarias ---
# cole_naturaleza: Público → 0, Privado → 1
df['es_privado'] = (df['cole_naturaleza'] == 'Privado').astype(float)
df.loc[df['cole_naturaleza'].isna(), 'es_privado'] = np.nan

# cole_area_ubicacion: RURAL → 0, URBANO → 1
df['es_urbano'] = (df['cole_area_ubicacion'] == 'URBANO').astype(float)
df.loc[df['cole_area_ubicacion'].isna(), 'es_urbano'] = np.nan

# --- Género: M → 1, F → 0 ---
df['genero_num'] = df['estu_genero'].map({'M': 1, 'F': 0})

In [ ]:
# Definir el conjunto de features que usará el modelo
FEATURES = [
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'indice_activos',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tieneautomovil',
    'fami_tienelavadora',
    'es_privado',
    'es_urbano',
    'genero_num',
]

TARGET = 'bajo_rendimiento'

df_model = df[FEATURES + [TARGET]].copy()
print(f"Shape del dataset de modelamiento: {df_model.shape}")
df_model.head(3)

### 2.2 Datos faltantes

In [ ]:
missing = df_model.isnull().mean().sort_values(ascending=False) * 100
missing_table = PrettyTable()
missing_table.field_names = ["Feature", "% Faltante"]
for col, pct in missing.items():
    missing_table.add_row([col, f"{pct:.2f}%"])
print(missing_table)

### 2.3 Analysis

In [ ]:
# Estrato vs bajo_rendimiento.
# Se espera una relación inversa: estratos bajos → mayor proporción de bajo rendimiento.
# Confirmarlo valida que el feature aporta poder predictivo.
estrato_risk = (
    df_model.dropna(subset=['estrato_num'])
    .groupby('estrato_num')[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'tasa_riesgo'})
)

estrato_risk['estrato_num'] = estrato_risk['estrato_num'].astype(str)

chart_estrato = alt.Chart(estrato_risk).mark_bar().encode(
    x=alt.X('estrato_num:O', title='Estrato', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
    color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
).properties(
    width=350, height=280,
    title='Tasa de bajo rendimiento por estrato socioeconómico'
)

chart_estrato

In [ ]:
# Índice de activos vs tasa de riesgo.
# El P1 confirmó que a mayor índice de activos, mayor puntaje.
# Debería verse una relación inversa con la tasa de bajo rendimiento.

activos_risk = (
    df_model.dropna(subset=['indice_activos'])
    .groupby('indice_activos')[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'tasa_riesgo'})
)
activos_risk['indice_activos'] = activos_risk['indice_activos'].astype(str)

chart_activos = alt.Chart(activos_risk).mark_bar().encode(
    x=alt.X('indice_activos:O', title='Índice de activos del hogar (0 = ninguno, 4 = todos)',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
    color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
).properties(
    width=350, height=280,
    title='Tasa de bajo rendimiento por índice de activos del hogar'
)

chart_activos

In [ ]:
# Educación de la madre vs tasa de riesgo.
# Se espera gradiente decreciente de riesgo a medida que aumenta el nivel educativo.
edu_risk = (
    df_model.dropna(subset=['edu_madre_num'])
    .groupby('edu_madre_num')[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'tasa_riesgo'})
)

edu_labels = {
    0: 'Ninguno', 1: 'Prim. inc.', 2: 'Prim. comp.',
    3: 'Sec. inc.', 4: 'Sec. comp.', 5: 'Téc. inc.',
    6: 'Téc. comp.', 7: 'Prof. inc.', 8: 'Prof. comp.', 9: 'Postgrado'
}
edu_risk['nivel'] = edu_risk['edu_madre_num'].map(edu_labels)

chart_edu = alt.Chart(edu_risk).mark_bar().encode(
    x=alt.X('edu_madre_num:O', title='Nivel educativo de la madre',
            axis=alt.Axis(labelExpr="{'0':'Ninguno','1':'Prim.inc','2':'Prim.comp','3':'Sec.inc','4':'Sec.comp','5':'Téc.inc','6':'Téc.comp','7':'Prof.inc','8':'Prof.comp','9':'Postgrado'}[datum.label]",
                          labelAngle=-35)),
    y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
    tooltip=['nivel:N', alt.Tooltip('tasa_riesgo:Q', format='.1%')],
    color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
).properties(
    width=400, height=280,
    title='Tasa de bajo rendimiento por nivel educativo de la madre'
)

chart_edu

In [ ]:
# Naturaleza del colegio y zona vs tasa de riesgo.
# El P1 mostró que privado > público en puntaje, y que la brecha
# rural-urbana no es uniforme. Se verifica si estos features discriminan la clase.
def tasa_por_binaria(col, label_0, label_1):
    tmp = (
        df_model.dropna(subset=[col])
        .groupby(col)[TARGET].mean()
        .reset_index()
        .rename(columns={col: 'valor', TARGET: 'tasa_riesgo'})
    )
    tmp['etiqueta'] = tmp['valor'].map({0.0: label_0, 1.0: label_1})
    return tmp

df_priv = tasa_por_binaria('es_privado', 'Público', 'Privado')
df_urb  = tasa_por_binaria('es_urbano', 'Rural', 'Urbano')

def bar_binaria(data, titulo):
    return alt.Chart(data).mark_bar().encode(
        x=alt.X('etiqueta:N', title='', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
        color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
    ).properties(width=200, height=260, title=titulo)

(bar_binaria(df_priv, 'Público vs Privado') | bar_binaria(df_urb, 'Rural vs Urbano'))

### 2.4 Correlación de features con el target

In [ ]:
# Mapa de calor de correlaciones entre todos los features.
# Permite identificar multicolinealidad: si dos features están muy correlacionados
# entre sí (ej. indice_activos y sus componentes), el modelo puede beneficiarse
# de usar solo uno de los dos grupos.

corr_matrix = df_model.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Matriz de correlación - features del modelo Q2', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Pregunta de negocio

### *¿Puede predecirse el nivel de desempeño en inglés de un estudiante (A−, A1, A2, B1, B+) a partir de su perfil académico, ¿socioeconómico y de género?*